### Download and Extract MVTec AD Dataset

To resolve the `FileNotFoundError`, we need to download the MVTec Anomaly Detection dataset. This code block will download the dataset archive and extract it to the specified `DATASET_DIR` (`./mvtec_anomaly_detection`). It includes a check to prevent re-downloading if the directory already exists.

In [ ]:
import os
import shutil
from pathlib import Path

# The project root is now the current working directory set by FiGC8Kiid1Vf
PROJECT_ROOT = Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

ARCHIVE_NAME = "mvtec_anomaly_detection.tar.xz"
ARCHIVE_PATH = PROJECT_ROOT / ARCHIVE_NAME
TEMP_EXTRACT_DIR = PROJECT_ROOT / "mvtec_temp_extracted"
DATASET_SUBDIR_IN_ARCHIVE = "mvtec_anomaly_detection"
ACTUAL_DATASET_PATH_IN_TEMP = TEMP_EXTRACT_DIR / DATASET_SUBDIR_IN_ARCHIVE

# Ensure data/raw exists
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Check if the dataset is already extracted and moved to data/raw
if any(RAW_DATA_DIR.iterdir()):
    print(f"Dataset already appears to be in {RAW_DATA_DIR}. Skipping extraction.")
    # Clean up any temporary extraction directories if they exist
    if TEMP_EXTRACT_DIR.exists():
        print(f"Cleaning up temporary extraction directory: {TEMP_EXTRACT_DIR}")
        shutil.rmtree(TEMP_EXTRACT_DIR)
else:
    if not ARCHIVE_PATH.exists():
        raise FileNotFoundError(
            f"MVTec archive '{ARCHIVE_PATH}' not found. "
            "Please ensure you have manually downloaded 'mvtec_anomaly_detection.tar.xz' "
            "from https://www.mvtec.com/company/research/datasets/mvtec-dataset-download/ "
            f"and uploaded it directly into your Google Drive folder: '{PROJECT_ROOT}'. "
            "Then re-run this cell."
        )
    else:
        print(f"Extracting {ARCHIVE_NAME} from {ARCHIVE_PATH} to {TEMP_EXTRACT_DIR}...")
        TEMP_EXTRACT_DIR.mkdir(parents=True, exist_ok=True) # Ensure temp extract dir exists
        !tar -xvf {ARCHIVE_PATH} -C {TEMP_EXTRACT_DIR}
        print("Extraction complete.")

        # Move contents from temporary extraction to data/raw
        if ACTUAL_DATASET_PATH_IN_TEMP.exists():
            print(f"Moving dataset contents from {ACTUAL_DATASET_PATH_IN_TEMP} to {RAW_DATA_DIR}...")
            for item in ACTUAL_DATASET_PATH_IN_TEMP.iterdir():
                shutil.move(str(item), str(RAW_DATA_DIR / item.name))
            shutil.rmtree(TEMP_EXTRACT_DIR) # Remove the temporary extracted folder
            print("Dataset moved successfully.")
        else:
            print(f"Warning: Expected dataset directory '{ACTUAL_DATASET_PATH_IN_TEMP}' not found after extraction. Check archive contents.")
            print(f"Contents of {TEMP_EXTRACT_DIR}:")
            !ls {TEMP_EXTRACT_DIR}

# Verify the contents of the RAW_DATA_DIR
print(f"Contents of {RAW_DATA_DIR}:")
!ls {RAW_DATA_DIR}

FileNotFoundError: MVTec archive '/content/drive/MyDrive/VisionBench/mvtec_anomaly_detection.tar.xz' not found. Please ensure you have manually downloaded 'mvtec_anomaly_detection.tar.xz' from https://www.mvtec.com/company/research/datasets/mvtec-dataset-download/ and uploaded it directly into your Google Drive folder: '/content/drive/MyDrive/VisionBench'. Then re-run this cell.

### Verify MVTec Archive in Google Drive

It appears the `mvtec_anomaly_detection.tar.xz` archive was not found at the expected location. Please ensure you have uploaded it directly to your `VisionBench` folder in Google Drive.

To verify, you can manually check your Google Drive folder structure (e.g., in a web browser: `My Drive/VisionBench/mvtec_anomaly_detection.tar.xz`).

Then, run the following cell to confirm the file is visible in this Colab environment within the expected directory.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd() # Should be /content/drive/MyDrive/VisionBench
ARCHIVE_PATH = PROJECT_ROOT / "mvtec_anomaly_detection.tar.xz"

print(f"Checking for archive at: {ARCHIVE_PATH}")
if ARCHIVE_PATH.exists():
    print(f"SUCCESS: MVTec archive found! Size: {ARCHIVE_PATH.stat().st_size / (1024*1024):.2f} MB")
    print("You can now proceed to run the extraction cell (a58f5a50) again.")
else:
    print("ERROR: MVTec archive NOT FOUND. Please ensure you uploaded 'mvtec_anomaly_detection.tar.xz' to:")
    print(f"  '{PROJECT_ROOT}'")
    print("If it's in a subfolder, move it directly into 'VisionBench'.")

print("\nContents of current directory (for debugging):")
!ls -F {PROJECT_ROOT}


Checking for archive at: /content/drive/MyDrive/VisionBench/mvtec_anomaly_detection.tar.xz
ERROR: MVTec archive NOT FOUND. Please ensure you uploaded 'mvtec_anomaly_detection.tar.xz' to:
  '/content/drive/MyDrive/VisionBench'
If it's in a subfolder, move it directly into 'VisionBench'.

Contents of current directory (for debugging):
data/  src/


### Next Steps:

1.  **Verify the file:** Run the code cell above this markdown cell to check if `mvtec_anomaly_detection.tar.xz` is now found.
2.  **Re-run `a58f5a50`:** If the verification confirms the archive is present, go back and run cell `a58f5a50` again. This should now successfully extract the dataset.
3.  **Continue the workflow:** After `a58f5a50` completes successfully, proceed with the rest of your notebook, specifically ensuring to run `GSVrfTRPXMaA` (to create the index script) and `hncsq9L8XO3m` (to build the index) before attempting training.

In [ ]:
import os
import glob
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm  # Hugging Face backend for computer vision models

# ==========================================
# 1. Custom Dataset Handler for MVTec AD
# ==========================================
class MVTecBinaryDataset(Dataset):
    """
    Custom Dataset to parse MVTec AD directory structure.
    Label 0: 'good' (Normal)
    Label 1: 'defective' (Any anomaly type)
    """
    def __init__(self, root_dir, category="bottle", split="train", transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []

        base_path = os.path.join(root_dir, category, split)

        # Iterate over subdirectories in the split (e.g., 'good', 'broken_large', etc.)
        for folder in os.listdir(base_path):
            folder_path = os.path.join(base_path, folder)
            if not os.path.isdir(folder_path):
                continue

            # Label 'good' as 0, and any defect folder as 1
            label = 0 if folder == "good" else 1

            # Find all image paths inside this folder
            for ext in ('*.png', '*.JPEG', '*.jpg'):
                for img_path in glob.glob(os.path.join(folder_path, ext)):
                    self.image_paths.append(img_path)
                    self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


# ==========================================
# 2. Data Transforms (Preprocessing & Augmentation)
# ==========================================
# ResNet models expect standard ImageNet normalization values
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ]),
}


# ==========================================
# 3. Model Architecture & Layer Freezing
# ==========================================
def build_resnet_model(num_classes=2, freeze_until='layer3'):
    """
    Loads ResNet-50 from Hugging Face timm repository,
    freezes lower layers, and updates the classifier.
    """
    # Load pre-trained ResNet-50 from Hugging Face (timm hub)
    model = timm.create_model('resnet50', pretrained=True, num_classes=num_classes)

    # 1. Freeze all initial backbone parameters first
    for param in model.parameters():
        param.requires_grad = False

    # 2. Selectively unfreeze deeper layers (layer3, layer4, fc)
    # Higher-level visual features like complex textures/anomalies are captured in deep layers.
    unfreeze = False
    for name, child in model.named_children():
        if name == freeze_until:
            unfreeze = True  # Start unfreezing from specified layer onwards
        if unfreeze:
            for param in child.parameters():
                param.requires_grad = True

    # Always ensure final classification head is trainable
    for param in model.fc.parameters():
        param.requires_grad = True

    return model


# ==========================================
# 4. Training Loop
# ==========================================
def train_model(model, dataloaders, criterion, optimizer, device, num_epochs=10):
    model.to(device)

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 20)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                # Forward pass
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward pass + optimize only during training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)

            print(f"{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

    return model


# ==========================================
# 5. Execution Script
# ==========================================
if __name__ == "__main__":
    # Specify your local dataset path
    DATASET_DIR = "./data/raw"  # Updated path to where the dataset is moved
    CATEGORY = "bottle"  # Options: bottle, cable, capsule, carpet, grid, hazelnut, etc.

    # Check GPU availability
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load Datasets
    # Note: MVTec AD standard test split contains defective samples.
    train_dataset = MVTecBinaryDataset(DATASET_DIR, category=CATEGORY, split="train", transform=data_transforms['train'])
    test_dataset = MVTecBinaryDataset(DATASET_DIR, category=CATEGORY, split="test", transform=data_transforms['val'])

    dataloaders = {
        'train': DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2),
        'val': DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)
    }

    # Instantiate Model with Layer Freezing
    model = build_resnet_model(num_classes=2, freeze_until='layer3')

    # Inspect trainable vs frozen parameters
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Parameters: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%) unskilled.")

    # Loss Function & Optimizer (Filter optimizer to only pass trainable parameters)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

    # Train Model
    trained_model = train_model(model, dataloaders, criterion, optimizer, device, num_epochs=10)

    # Save Model Weights
    torch.save(trained_model.state_dict(), f"resnet50_{CATEGORY}_defect_detector.pth")

Using device: cpu


FileNotFoundError: [Errno 2] No such file or directory: './data/raw/bottle/train'

In [ ]:
# ============================================================
# VISIONBENCH — FINAL DRAEM TRAINING CELL
# ============================================================

from pathlib import Path
import sys
import pandas as pd
import torch
import shutil # Import shutil for moving directories

PROJECT = Path.cwd() # Set project root to current working directory
RAW = PROJECT / "data" / "raw"

# ------------------------------------------------------------
# Ensure project directories exist and set up Python path
# ------------------------------------------------------------
PROJECT.mkdir(parents=True, exist_ok=True)
RAW.mkdir(parents=True, exist_ok=True) # Ensure RAW exists if not already by previous cell

# We assume dataset moving is handled by cell a58f5a50
# Removed: dataset moving logic as it's now in a58f5a50

# 1. Make project packages visible to Python
sys.path.insert(0, str(PROJECT))

for folder in [
    PROJECT / "src",
    PROJECT / "src" / "data",
    PROJECT / "src" / "models",
    PROJECT / "src" / "training",
]:
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "__init__.py").touch()

# 2. Create the missing MVTec dataset module
dataset_code = '''
from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import Dataset
from PIL import Image


class MVTecDataset(
    Dataset
):

    def __init__(
        self,
        index_file,
        split="train",
        transform=None,
        return_mask=False
    ):
        self.index_file = Path(index_file)
        # self.project_root should correctly infer from index_file path
        # Assuming index_file is something like PROJECT/data/mvtec_index.csv
        # so self.project_root would be PROJECT (Path(index_file).parent.parent)
        self.project_root = self.index_file.parent.parent

        self.data = pd.read_csv(self.index_file)
        self.data = self.data[
            self.data["split"] == split
        ].reset_index(drop=True)

        self.transform = transform
        self.return_mask = return_mask

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        row = self.data.iloc[index]

        image_path = self.project_root / row["image_path"]
        image = Image.open(image_path).convert("RGB")

        label = torch.tensor(
            int(row["label"]),
            dtype=torch.long
        )

        mask = None

        if (
            self.return_mask
            and pd.notna(row["mask_path"])
            and row["mask_path"] != ""
        ):
            mask_path = self.project_root / row["mask_path"]
            mask = Image.open(mask_path).convert("L")

        if self.transform:
            image = self.transform(image)

        result = {
            "image": image,
            "label": label,
            "category": row["category"],
            "defect_type": row["defect_type"],
            "image_path": str(image_path),
        }

        if self.return_mask:
            result["mask"] = mask

        return result
'''

(PROJECT / "src" / "data" / "dataset.py").write_text(dataset_code)

# 3. Build MVTec index (logic below is for explicit index creation, but pJKWrZjhco2O also triggers it.
# This part is probably redundant if pJKWrZjhco2O is used, but keeping it ensures index is built if this cell is run standalone)

rows = []

# Check if RAW is populated before attempting to list categories
if RAW.exists() and any(RAW.iterdir()):
    categories = sorted([
        p.name for p in RAW.iterdir()
        if p.is_dir()
    ])

    for category in categories:

        category_dir = RAW / category
        train_dir = category_dir / "train"
        test_dir = category_dir / "test"
        gt_dir = category_dir / "ground_truth"

        # Normal training images
        good_dir = train_dir / "good"

        if good_dir.exists():

            for img in sorted(good_dir.glob("*.png")):

                rows.append({
                    "category": category,
                    "split": "train",
                    "defect_type": "good",
                    "image_path": str(
                        img.relative_to(PROJECT)
                    ).replace("\\", "/"),
                    "mask_path": "",
                    "label": 0
                })

        # Test images
        if test_dir.exists():

            for defect_dir in sorted(test_dir.iterdir()):

                if not defect_dir.is_dir():
                    continue

                defect_type = defect_dir.name

                for img in sorted(defect_dir.glob("*.png")):

                    label = 0 if defect_type == "good" else 1
                    mask = ""

                    if label == 1:

                        mask_file = (
                            gt_dir
                            / defect_type
                            / f"{img.stem}_mask.png"
                        )

                        if mask_file.exists():
                            mask = str(
                                mask_file.relative_to(PROJECT)
                            ).replace("\\", "/")

                    rows.append({
                        "category": category,
                        "split": "test",
                        "defect_type": defect_type,
                        "image_path": str(
                            img.relative_to(PROJECT)
                        ).replace("\\", "/"),
                        "mask_path": mask,
                        "label": label
                    })
else:
    print(f"Warning: '{RAW}' is empty. MVTec index will be generated with 0 entries.")
    categories = []

index_path = PROJECT / "data" / "mvtec_index.csv"
index_path.parent.mkdir(parents=True, exist_ok=True)

pd.DataFrame(rows).to_csv(
    index_path,
    index=False
)

print("MVTec index created")
print("Categories:", len(categories))
print("Total images:", len(rows))

# 4. GPU check

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Stop here.")

print("GPU:", torch.cuda.get_device_name(0))

# 5. Import DRAEM modules BEFORE training

from src.data.dataset import MVTecDataset
from src.models.draem import DRAEM

print("DRAEM modules imported successfully")

# 6. Start training

train_script = PROJECT / "src" / "training" / "train_draem.py"

if not train_script.exists():
    raise FileNotFoundError(
        f"Missing training script: {train_script}"
    )

print("\n==========================================")
print("STARTING DRAEM TRAINING")
print("==========================================\n")

# Execute training inside the same Python environment
exec(
    compile(
        train_script.read_text(),
        str(train_script),
        "exec"
    ),
    {
        "__name__": "__main__",
        "__file__": str(train_script),
        "__package__": None,
    }
)

print("\n==========================================")
print("DRAEM TRAINING FINISHED")
print("==========================================")

MVTec index created
Categories: 0
Total images: 0


RuntimeError: CUDA is not available. Stop here.

In [ ]:
from pathlib import Path
import tarfile
from google.colab import files

PROJECT = Path("/content/VisionBench")
RAW = PROJECT / "data" / "raw"
ARCHIVE = Path("/content/mvtec_anomaly_detection.tar.xz")

RAW.mkdir(parents=True, exist_ok=True)

# If the dataset archive is not already in this runtime, upload it
if not ARCHIVE.exists():
    print("MVTec archive not found in this Colab runtime.")
    print("Select: mvtec_anomaly_detection.tar.xz")
    uploaded = files.upload()

    uploaded_file = Path(next(iter(uploaded.keys())))
    if uploaded_file.name != ARCHIVE.name:
        raise RuntimeError(
            f"Wrong file uploaded: {uploaded_file.name}"
        )

    ARCHIVE = uploaded_file

# Extract
print("Extracting MVTec...")
with tarfile.open(ARCHIVE, "r:xz") as tar:
    tar.extractall(RAW)

# Verify
categories = sorted(
    p.name for p in RAW.iterdir()
    if p.is_dir()
)

print("\nMVTec extraction complete.")
print("Categories:", len(categories))
print(categories)

MVTec archive not found in this Colab runtime.
Select: mvtec_anomaly_detection.tar.xz


KeyboardInterrupt: 

In [ ]:
import os

# Check current working directory
print("Current Working Directory:", os.getcwd())

# List files in the current directory to verify where your data folder is
print("Files in current folder:", os.listdir('.'))

Current Working Directory: /content
Files in current folder: ['.config', 'VisionBench', 'sample_data']


In [ ]:
import os
os.chdir('/content/VisionBench')

# Verify you are inside VisionBench now
print("Updated Directory:", os.getcwd())
print("Files inside VisionBench:", os.listdir('.'))

Updated Directory: /content/VisionBench
Files inside VisionBench: ['src', 'data']


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, models
import pandas as pd
from PIL import Image

# 1. Dataset Loader mapped to your VisionBench structure
class MVTecTrainDataset(torch.utils.data.Dataset):
    def __init__(self, csv_path, category, transform=None):
        df = pd.read_csv(csv_path)
        self.df = df[(df['category'] == category) & (df['split'] == 'train')].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# 2. On-GPU Synthetic Anomaly Generator
class GPUAnomalyGenerator:
    def __init__(self, prob=0.5):
        self.prob = prob

    def __call__(self, img_batch):
        B, C, H, W = img_batch.shape
        anom_batch = img_batch.clone()
        mask_batch = torch.zeros((B, 1, H, W), device=img_batch.device)

        for i in range(B):
            if torch.rand(1).item() < self.prob:
                rh = torch.randint(int(H * 0.15), int(H * 0.45), (1,)).item()
                rw = torch.randint(int(W * 0.15), int(W * 0.45), (1,)).item()
                top = torch.randint(0, H - rh, (1,)).item()
                left = torch.randint(0, W - rw, (1,)).item()

                mask_batch[i, 0, top:top+rh, left:left+rw] = 1.0

                mode = torch.randint(0, 3, (1,)).item()
                if mode == 0:
                    anom_batch[i, :, top:top+rh, left:left+rw] = torch.rand((C, rh, rw), device=img_batch.device)
                elif mode == 1:
                    anom_batch[i, :, top:top+rh, left:left+rw] *= 0.1
                else:
                    anom_batch[i, :, top:top+rh, left:left+rw] = torch.clamp(
                        anom_batch[i, :, top:top+rh, left:left+rw] + 0.7, 0.0, 1.0
                    )
        return anom_batch, mask_batch

# 3. Custom DRAEM-style Architecture
class CustomDRAEM(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv3 = nn.Sequential(nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True))
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv2 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True))
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv1 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True))
        self.up1 = nn.ConvTranspose2d(64, 32, 4, stride=4)
        self.out_recon = nn.Conv2d(32, 3, 3, padding=1)

        self.segmentor = nn.Sequential(
            nn.Conv2d(6, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 1, 3, padding=1)
        )

    def forward(self, x):
        x1 = self.layer1(self.stem(x))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)

        d = self.up4(x4)
        d = self.conv3(torch.cat([d, x3], dim=1))
        d = self.up3(d)
        d = self.conv2(torch.cat([d, x2], dim=1))
        d = self.up2(d)
        d = self.conv1(torch.cat([d, F.interpolate(x1, size=d.shape[2:])], dim=1))
        d = self.up1(d)
        recon = torch.sigmoid(self.out_recon(d))

        seg_input = torch.cat([x, recon], dim=1)
        seg_logits = self.segmentor(seg_input)

        return recon, seg_logits

# 4. Training Engine
def run_draem_training(category="bottle", epochs=5, batch_size=16):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dataset = MVTecTrainDataset(csv_path="data/mvtec_index.csv", category=category, transform=train_transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

    model = CustomDRAEM().to(device)
    anomaly_gen = GPUAnomalyGenerator(prob=0.5)

    l2_loss = nn.MSELoss()
    bce_loss = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

    print(f"--- Training Custom DRAEM: [{category.upper()}] on {device} ---")

    for epoch in range(1, epochs + 1):
        model.train()
        total_epoch_loss, recon_epoch_loss, seg_epoch_loss = 0.0, 0.0, 0.0

        for clean_imgs in loader:
            clean_imgs = clean_imgs.to(device)
            anom_imgs, gt_masks = anomaly_gen(clean_imgs)

            optimizer.zero_grad()
            recon_imgs, seg_logits = model(anom_imgs)

            loss_l2 = l2_loss(recon_imgs, clean_imgs)
            loss_bce = bce_loss(seg_logits, gt_masks)
            loss = loss_l2 + (2.0 * loss_bce)

            loss.backward()
            optimizer.step()

            total_epoch_loss += loss.item()
            recon_epoch_loss += loss_l2.item()
            seg_epoch_loss += loss_bce.item()

        num_batches = len(loader)
        print(f"Epoch [{epoch:02d}/{epochs:02d}] | Total Loss: {total_epoch_loss/num_batches:.4f} | Recon L2: {recon_epoch_loss/num_batches:.4f} | Seg BCE: {seg_epoch_loss/num_batches:.4f}")

    return model

# Run test training batch
trained_model = run_draem_training(category="bottle", epochs=5)

EmptyDataError: No columns to parse from file

In [ ]:
import os

CSV_PATH = "data/mvtec_index.csv"

if os.path.exists(CSV_PATH):
    file_size = os.path.getsize(CSV_PATH)
    print(f"File '{CSV_PATH}' exists.")
    print(f"Size of the file: {file_size} bytes.")

    if file_size == 0:
        print("\n[CONFIRMED] The file is empty (0 bytes).")
    else:
        print("\n--- Start of File ---")
        try:
            with open(CSV_PATH, 'r') as f:
                print(f.read(200)) # Print first 200 characters
        except Exception as e:
            print(f"Could not read content: {e}")
        print("--- End of Preview ---")
else:
    print(f"[ERROR] The file '{CSV_PATH}' was not found.")

File 'data/mvtec_index.csv' exists.
Size of the file: 1 bytes.

--- Start of File ---


--- End of Preview ---


In [ ]:
!python src/data/build_index.py

python3: can't open file '/content/VisionBench/src/data/build_index.py': [Errno 2] No such file or directory


In [ ]:
import os
print("Current Working Directory:", os.getcwd())

Current Working Directory: /content/VisionBench


In [ ]:
import os
import pandas as pd
from pathlib import Path

# Create directory structure if missing
os.makedirs("src/data", exist_ok=True)

# Write the index generator code to src/data/build_index.py
build_script_content = """import os
import pandas as pd
from pathlib import Path

def generate_mvtec_index(data_dir="data/raw", output_csv="data/mvtec_index.csv"):
    data_dir = Path(data_dir)
    records = []

    if not data_dir.exists():
        print(f"Error: Directory {data_dir} does not exist.")
        return

    categories = [d.name for d in data_dir.iterdir() if d.is_dir()]

    for category in sorted(categories):
        cat_dir = data_dir / category

        # Process Train Split
        train_dir = cat_dir / "train" / "good"
        if train_dir.exists():
            for img_path in train_dir.glob("*.png"):
                records.append({
                    "category": category,
                    "split": "train",
                    "defect_type": "good",
                    "image_path": str(img_path),
                    "mask_path": "",
                    "label": 0
                })

        # Process Test Split
        test_dir = cat_dir / "test"
        ground_truth_dir = cat_dir / "ground_truth"

        if test_dir.exists():
            for defect_dir in test_dir.iterdir():
                if defect_dir.is_dir():
                    defect_type = defect_dir.name
                    label = 0 if defect_type == "good" else 1

                    for img_path in defect_dir.glob("*.png"):
                        mask_path_str = ""
                        if label == 1:
                            mask_name = img_path.stem + "_mask.png"
                            target_mask = ground_truth_dir / defect_type / mask_name
                            if target_mask.exists():
                                mask_path_str = str(target_mask)

                        records.append({
                            "category": category,
                            "split": "test",
                            "defect_type": defect_type,
                            "image_path": str(img_path),
                            "mask_path": mask_path_str,
                            "label": label
                        })

    df = pd.DataFrame(records)
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"Successfully generated '{output_csv}' with {len(df)} total image entries.")

if __name__ == "__main__":
    generate_mvtec_index()
"""

with open("src/data/build_index.py", "w") as f:
    f.write(build_script_content)

print("Created src/data/build_index.py successfully.")

Created src/data/build_index.py successfully.


In [ ]:
!python src/data/build_index.py

Successfully generated 'data/mvtec_index.csv' with 0 total image entries.


In [ ]:
import os
print("Direct contents of data/raw:")
print(os.listdir("data/raw"))

# Check inside the first category folder
first_item = os.listdir("data/raw")[0]
first_path = os.path.join("data/raw", first_item)
if os.path.isdir(first_path):
    print(f"\nContents inside data/raw/{first_item}:")
    print(os.listdir(first_path))

Direct contents of data/raw:
[]


IndexError: list index out of range

In [ ]:
import os
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()
INDEX_CSV_PATH = PROJECT_ROOT / "data" / "mvtec_index.csv"
BUILD_INDEX_SCRIPT_PATH = PROJECT_ROOT / "src" / "data" / "build_index.py"


# Check that the index file exists and is not empty
if INDEX_CSV_PATH.exists() and INDEX_CSV_PATH.stat().st_size > 0:
    df = pd.read_csv(INDEX_CSV_PATH)
    print(f"Index found with {len(df)} images across {df['category'].nunique()} categories.")
else:
    print("Index not found or is empty. Running index builder...")
    # First, verify the build script exists
    if not BUILD_INDEX_SCRIPT_PATH.exists():
        print(f"Error: Build index script not found at {BUILD_INDEX_SCRIPT_PATH}. Please ensure cell 'GSVrfTRPXMaA' was run successfully.")
    else:
        # Run the index builder script using its absolute path
        # Need to use PROJECT_ROOT here for current directory context
        !python {BUILD_INDEX_SCRIPT_PATH}
        # Re-check the index after running the script
        if INDEX_CSV_PATH.exists() and INDEX_CSV_PATH.stat().st_size > 0:
            df = pd.read_csv(INDEX_CSV_PATH)
            print(f"Index successfully created/updated with {len(df)} images across {df['category'].nunique()} categories.")
        else:
            print("Warning: Index builder ran, but 'mvtec_index.csv' is still empty or missing. Check previous outputs for dataset issues.")

EmptyDataError: No columns to parse from file

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Change directory to where your VisionBench project lives in Drive
# Replace 'VisionBench' with your actual Drive folder path if different
PROJECT_PATH = '/content/drive/MyDrive/VisionBench'

# Create the project directory if it does not exist
if not os.path.exists(PROJECT_PATH):
    os.makedirs(PROJECT_PATH)
    print(f"Created directory: {PROJECT_PATH}")

os.chdir(PROJECT_PATH)
print("Current Working Directory:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Created directory: /content/drive/MyDrive/VisionBench
Current Working Directory: /content/drive/MyDrive/VisionBench


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import roc_auc_score

# ---------------------------------------------------------
# 1. DRAEM Model Architecture
# ---------------------------------------------------------
class EncoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

    def forward(self, x):
        x0 = self.relu(self.bn1(self.conv1(x)))
        x1 = self.layer1(self.maxpool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return [x0, x1, x2, x3, x4]

class DecoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = nn.Sequential(nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU())
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU())
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec0 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.final = nn.Sequential(nn.Conv2d(64, 3, 3, padding=1), nn.Sigmoid())

    def forward(self, feats):
        x0, x1, x2, x3, x4 = feats
        d4 = self.dec3(torch.cat([self.up4(x4), x3], dim=1))
        d3 = self.dec2(torch.cat([self.up3(d4), x2], dim=1))
        d2 = self.dec1(torch.cat([self.up2(d3), x1], dim=1))
        d1 = self.dec0(torch.cat([self.up1(d2), x0], dim=1))
        return self.final(d1)

class DiscriminatorSegment(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(6, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU()
        )
        self.head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, orig, recon):
        x = torch.cat([orig, recon], dim=1)
        return self.head(self.stem(x))

class DRAEM(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = EncoderReconstruct()
        self.decoder = DecoderReconstruct()
        self.segmentor = DiscriminatorSegment()

    def forward(self, x):
        feats = self.encoder(x)
        recon = self.decoder(feats)
        seg_logits = self.segmentor(x, recon)
        return recon, seg_logits

# ---------------------------------------------------------
# 2. Synthetic Anomaly Dataset Generator
# ---------------------------------------------------------
def generate_perlin_noise_mask(height=224, width=224):
    scale = np.random.choice([16, 32, 64])
    h_small, w_small = max(1, height // scale), max(1, width // scale)
    noise = np.random.randn(h_small, w_small)
    noise = cv2.resize(noise, (width, height), interpolation=cv2.INTER_CUBIC)
    mask = (noise > 0.3).astype(np.float32)
    return torch.from_numpy(mask).unsqueeze(0)

class DRAEMTrainDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df[df['split'] == 'train'].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        anomaly_mask = generate_perlin_noise_mask(img.shape[1], img.shape[2])
        noise_source = torch.rand_like(img)
        aug_img = img * (1.0 - anomaly_mask) + noise_source * anomaly_mask

        return {"augmented_image": aug_img, "target_image": img, "anomaly_mask": anomaly_mask}

# ---------------------------------------------------------
# 3. Training & Evaluation Runner
# ---------------------------------------------------------
def train_and_eval_category(category_name, df, epochs=15, batch_size=8, lr=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cat_df = df[df['category'] == category_name]
    transform = T.Compose([T.Resize((224, 224)), T.ToTensor()])

    train_ds = DRAEMTrainDataset(cat_df, transform=transform)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = DRAEM().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    l2_loss = nn.MSELoss()
    bce_loss = nn.BCEWithLogitsLoss()

    model.train()
    for epoch in range(epochs):
        for batch in train_loader:
            aug = batch['augmented_image'].to(device)
            target = batch['target_image'].to(device)
            mask = batch['anomaly_mask'].to(device)

            optimizer.zero_grad()
            recon, seg_logits = model(aug)

            loss_rec = l2_loss(recon, target)
            loss_seg = bce_loss(seg_logits, mask)
            loss = loss_rec + 2.0 * loss_seg

            loss.backward()
            optimizer.step()

    model.eval()
    test_df = cat_df[cat_df['split'] == 'test']
    y_true, y_scores = [], []

    with torch.no_grad():
        for _, row in test_df.iterrows():
            img = Image.open(row['image_path']).convert("RGB")
            tensor_img = transform(img).unsqueeze(0).to(device)
            recon, seg_logits = model(tensor_img)
            img_score = torch.max(torch.sigmoid(seg_logits)).item()

            y_true.append(row['label'])
            y_scores.append(img_score)

    return roc_auc_score(y_true, y_scores)

# Execution
if os.path.exists("data/mvtec_index.csv"):
    df = pd.read_csv("data/mvtec_index.csv")
    categories = df['category'].unique()
    results = {}

    print(f"Starting DRAEM Baseline 7 training across {len(categories)} categories...\n")
    for cat in categories:
        auroc = train_and_eval_category(cat, df)
        results[cat] = auroc
        print(f"Category: {cat:<12} | AUROC: {auroc:.4f}")

    mean_auroc = sum(results.values()) / len(results)
    print(f"\nBaseline 7 (DRAEM) Mean AUROC: {mean_auroc:.4f}")
else:
    print("Run Cell [4] first to build data/mvtec_index.csv.")

Run Cell [4] first to build data/mvtec_index.csv.


In [ ]:
import os
import pandas as pd
from pathlib import Path

def build_index_recursive(data_root="data/raw", output_csv="data/mvtec_index.csv"):
    data_path = Path(data_root)
    records = []
    for train_dir in data_path.rglob("train"):
        cat_dir = train_dir.parent
        category = cat_dir.name
        good_train = train_dir / "good"
        if good_train.exists():
            for img_path in good_train.glob("*.png"):
                records.append({"category": category, "split": "train", "defect_type": "good", "image_path": str(img_path), "mask_path": "", "label": 0})
        test_dir = cat_dir / "test"
        gt_dir = cat_dir / "ground_truth"
        if test_dir.exists():
            for defect_dir in test_dir.iterdir():
                if defect_dir.is_dir():
                    defect_type = defect_dir.name
                    label = 0 if defect_type == "good" else 1
                    for img_path in defect_dir.glob("*.png"):
                        mask_path_str = ""
                        if label == 1 and gt_dir.exists():
                            mask_file = gt_dir / defect_type / f"{img_path.stem}_mask.png"
                            if mask_file.exists():
                                mask_path_str = str(mask_file)
                        records.append({"category": category, "split": "test", "defect_type": defect_type, "image_path": str(img_path), "mask_path": mask_path_str, "label": label})
    df = pd.DataFrame(records)
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"Index successfully built: {len(df)} images found.")

build_index_recursive()

Index successfully built: 0 images found.


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import roc_auc_score

# ---------------------------------------------------------
# 1. DRAEM Model Architecture
# ---------------------------------------------------------
class EncoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

    def forward(self, x):
        x0 = self.relu(self.bn1(self.conv1(x)))
        x1 = self.layer1(self.maxpool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return [x0, x1, x2, x3, x4]

class DecoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = nn.Sequential(nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU())
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU())
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec0 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.final = nn.Sequential(nn.Conv2d(64, 3, 3, padding=1), nn.Sigmoid())

    def forward(self, feats):
        x0, x1, x2, x3, x4 = feats
        d4 = self.dec3(torch.cat([self.up4(x4), x3], dim=1))
        d3 = self.dec2(torch.cat([self.up3(d4), x2], dim=1))
        d2 = self.dec1(torch.cat([self.up2(d3), x1], dim=1))
        d1 = self.dec0(torch.cat([self.up1(d2), x0], dim=1))
        return self.final(d1)

class DiscriminatorSegment(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(6, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU()
        )
        self.head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, orig, recon):
        x = torch.cat([orig, recon], dim=1)
        return self.head(self.stem(x))

class DRAEM(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = EncoderReconstruct()
        self.decoder = DecoderReconstruct()
        self.segmentor = DiscriminatorSegment()

    def forward(self, x):
        feats = self.encoder(x)
        recon = self.decoder(feats)
        seg_logits = self.segmentor(x, recon)
        return recon, seg_logits

# ---------------------------------------------------------
# 2. Synthetic Anomaly Dataset Generator
# ---------------------------------------------------------
def generate_perlin_noise_mask(height=224, width=224):
    scale = np.random.choice([16, 32, 64])
    h_small, w_small = max(1, height // scale), max(1, width // scale)
    noise = np.random.randn(h_small, w_small)
    noise = cv2.resize(noise, (width, height), interpolation=cv2.INTER_CUBIC)
    mask = (noise > 0.3).astype(np.float32)
    return torch.from_numpy(mask).unsqueeze(0)

class DRAEMTrainDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df[df['split'] == 'train'].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        anomaly_mask = generate_perlin_noise_mask(img.shape[1], img.shape[2])
        noise_source = torch.rand_like(img)
        aug_img = img * (1.0 - anomaly_mask) + noise_source * anomaly_mask

        return {"augmented_image": aug_img, "target_image": img, "anomaly_mask": anomaly_mask}

# ---------------------------------------------------------
# 3. Training & Evaluation Runner
# ---------------------------------------------------------
def train_and_eval_category(category_name, df, epochs=15, batch_size=8, lr=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cat_df = df[df['category'] == category_name]
    transform = T.Compose([T.Resize((224, 224)), T.ToTensor()])

    train_ds = DRAEMTrainDataset(cat_df, transform=transform)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = DRAEM().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    l2_loss = nn.MSELoss()
    bce_loss = nn.BCEWithLogitsLoss()

    model.train()
    for epoch in range(epochs):
        for batch in train_loader:
            aug = batch['augmented_image'].to(device)
            target = batch['target_image'].to(device)
            mask = batch['anomaly_mask'].to(device)

            optimizer.zero_grad()
            recon, seg_logits = model(aug)

            loss_rec = l2_loss(recon, target)
            loss_seg = bce_loss(seg_logits, mask)
            loss = loss_rec + 2.0 * loss_seg

            loss.backward()
            optimizer.step()

    model.eval()
    test_df = cat_df[cat_df['split'] == 'test']
    y_true, y_scores = [], []

    with torch.no_grad():
        for _, row in test_df.iterrows():
            img = Image.open(row['image_path']).convert("RGB")
            tensor_img = transform(img).unsqueeze(0).to(device)
            recon, seg_logits = model(tensor_img)
            img_score = torch.max(torch.sigmoid(seg_logits)).item()

            y_true.append(row['label'])
            y_scores.append(img_score)

    return roc_auc_score(y_true, y_scores)

# Execution
if os.path.exists("data/mvtec_index.csv"):
    df = pd.read_csv("data/mvtec_index.csv")
    categories = df['category'].unique()
    results = {}

    print(f"Starting DRAEM Baseline 7 training across {len(categories)} categories...\n")
    for cat in categories:
        auroc = train_and_eval_category(cat, df)
        results[cat] = auroc
        print(f"Category: {cat:<12} | AUROC: {auroc:.4f}")

    mean_auroc = sum(results.values()) / len(results)
    print(f"\nBaseline 7 (DRAEM) Mean AUROC: {mean_auroc:.4f}")
else:
    print("Run Cell [4] first to build data/mvtec_index.csv.")

EmptyDataError: No columns to parse from file

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import tarfile

archive_path = "/content/drive/MyDrive/VisionBench_Datasets/mvtec_anomaly_detection.tar.xz"
extract_dir = "data/raw"

os.makedirs(extract_dir, exist_ok=True)

if os.path.exists(archive_path):
    print("Found archive! Extracting dataset (this takes ~1-2 mins)...")
    with tarfile.open(archive_path, "r:xz") as tar:
        tar.extractall(path=extract_dir)
    print("Extraction Complete!")
else:
    print(f"Error: Could not find file at {archive_path}")

Found archive! Extracting dataset (this takes ~1-2 mins)...


/tmp/ipykernel_6401/2183957596.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


Extraction Complete!


In [ ]:
import os
import pandas as pd
from pathlib import Path

records = []
data_path = Path("data/raw")

for train_dir in data_path.rglob("train"):
    cat_dir = train_dir.parent
    category = cat_dir.name
    good_train = train_dir / "good"
    if good_train.exists():
        for img_path in good_train.glob("*.png"):
            records.append({"category": category, "split": "train", "defect_type": "good", "image_path": str(img_path), "mask_path": "", "label": 0})
    test_dir = cat_dir / "test"
    gt_dir = cat_dir / "ground_truth"
    if test_dir.exists():
        for defect_dir in test_dir.iterdir():
            if defect_dir.is_dir():
                defect_type = defect_dir.name
                label = 0 if defect_type == "good" else 1
                for img_path in defect_dir.glob("*.png"):
                    mask_path_str = ""
                    if label == 1 and gt_dir.exists():
                        mask_file = gt_dir / defect_type / f"{img_path.stem}_mask.png"
                        if mask_file.exists():
                            mask_path_str = str(mask_file)
                    records.append({"category": category, "split": "test", "defect_type": defect_type, "image_path": str(img_path), "mask_path": mask_path_str, "label": label})

df = pd.DataFrame(records)
os.makedirs("data", exist_ok=True)
df.to_csv("data/mvtec_index.csv", index=False)
print(f"SUCCESS: Index created with {len(df)} images!")

SUCCESS: Index created with 5354 images!


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F # Import F for interpolate
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import roc_auc_score

# ---------------------------------------------------------
# 1. DRAEM Subnetworks Architecture
# ---------------------------------------------------------
class EncoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

    def forward(self, x):
        x0 = self.relu(self.bn1(self.conv1(x))) # 112x112
        x1 = self.layer1(self.maxpool(x0))     # 56x56
        x2 = self.layer2(x1)                   # 28x28
        x3 = self.layer3(x2)                   # 14x14
        x4 = self.layer4(x3)                   # 7x7
        return [x0, x1, x2, x3, x4]

class DecoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = nn.Sequential(nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU()) # Output 14x14
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU()) # Output 28x28
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU()) # Output 56x56
        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2) # Output 112x112
        self.dec0 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU()) # Output 112x112

        # Added an additional upsampling layer to reach 224x224
        self.up0 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2) # Upsamples 112x112 to 224x224
        self.final_conv = nn.Sequential(nn.Conv2d(32, 3, 3, padding=1), nn.Sigmoid()) # Final conv with corrected input channels

    def forward(self, feats):
        x0, x1, x2, x3, x4 = feats
        d4 = self.dec3(torch.cat([self.up4(x4), x3], dim=1))
        d3 = self.dec2(torch.cat([self.up3(d4), x2], dim=1))
        d2 = self.dec1(torch.cat([self.up2(d3), x1], dim=1))
        d1 = self.dec0(torch.cat([self.up1(d2), x0], dim=1))

        # Perform the final upsampling step
        d0 = self.up0(d1)
        return self.final_conv(d0)

class DiscriminatorSegment(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(6, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU()
        )
        self.head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, orig, recon):
        x = torch.cat([orig, recon], dim=1) # Now orig and recon will have matching spatial dimensions
        return self.head(self.stem(x))

class DRAEM(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = EncoderReconstruct()
        self.decoder = DecoderReconstruct()
        self.segmentor = DiscriminatorSegment()

    def forward(self, x):
        feats = self.encoder(x)
        recon = self.decoder(feats)
        seg_logits = self.segmentor(x, recon)
        return recon, seg_logits

# ---------------------------------------------------------
# 2. Synthetic Anomaly Generation Dataset
# ---------------------------------------------------------
def generate_perlin_noise_mask(height=224, width=224):
    scale = np.random.choice([16, 32, 64])
    h_small, w_small = max(1, height // scale), max(1, width // scale)
    noise = np.random.randn(h_small, w_small)
    noise = cv2.resize(noise, (width, height), interpolation=cv2.INTER_CUBIC)
    mask = (noise > 0.3).astype(np.float32)
    return torch.from_numpy(mask).unsqueeze(0)

class DRAEMTrainDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df[df['split'] == 'train'].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        anomaly_mask = generate_perlin_noise_mask(img.shape[1], img.shape[2])
        noise_source = torch.rand_like(img)
        aug_img = img * (1.0 - anomaly_mask) + noise_source * anomaly_mask

        return {"augmented_image": aug_img, "target_image": img, "anomaly_mask": anomaly_mask}

# ---------------------------------------------------------
# 3. Training & Evaluation Pipeline
# ---------------------------------------------------------
def train_and_eval_category(category_name, df, epochs=15, batch_size=8, lr=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cat_df = df[df['category'] == category_name]
    transform = T.Compose([T.Resize((224, 224)), T.ToTensor()])

    train_ds = DRAEMTrainDataset(cat_df, transform=transform)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = DRAEM().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    l2_loss = nn.MSELoss()
    bce_loss = nn.BCEWithLogitsLoss()

    # Training
    model.train()
    for epoch in range(epochs):
        for batch in train_loader:
            aug = batch['augmented_image'].to(device)
            target = batch['target_image'].to(device)
            mask = batch['anomaly_mask'].to(device)

            optimizer.zero_grad()
            recon, seg_logits = model(aug)

            loss_rec = l2_loss(recon, target)
            loss_seg = bce_loss(seg_logits, mask)
            loss = loss_rec + 2.0 * loss_seg

            loss.backward()
            optimizer.step()

    # Evaluation on Test Split
    model.eval()
    test_df = cat_df[cat_df['split'] == 'test']
    y_true, y_scores = [], []

    with torch.no_grad():
        for _, row in test_df.iterrows():
            img = Image.open(row['image_path']).convert("RGB")
            tensor_img = transform(img).unsqueeze(0).to(device)
            recon, seg_logits = model(tensor_img)
            img_score = torch.max(torch.sigmoid(seg_logits)).item()

            y_true.append(row['label'])
            y_scores.append(img_score)

    return roc_auc_score(y_true, y_scores)

# Execute Across All 15 MVTec AD Categories
df = pd.read_csv("data/mvtec_index.csv")
categories = df['category'].unique()
results = {}

print(f"Starting DRAEM Baseline 7 training across {len(categories)} categories...\n")
for cat in categories:
    auroc = train_and_eval_category(cat, df)
    results[cat] = auroc
    print(f"Category: {cat:<12} | AUROC: {auroc:.4f}")

mean_auroc = sum(results.values()) / len(results)
print(f"\n=========================================")
print(f"Baseline 7 (DRAEM) Mean AUROC: {mean_auroc:.4f}")
print(f"=========================================")

Starting DRAEM Baseline 7 training across 15 categories...



In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import roc_auc_score

# Check data index exists
if not os.path.exists("data/mvtec_index.csv"):
    print("Index missing! Run your dataset extraction cell first.")
else:
    df = pd.read_csv("data/mvtec_index.csv")
    categories = df['category'].unique()
    results = {}

    print(f"Starting FAST DRAEM Baseline 7 training (3 Epochs per Category)...\n")
    for cat in categories:
        auroc = train_and_eval_category(cat, df, epochs=3)
        results[cat] = auroc
        print(f"Category: {cat:<12} | AUROC: {auroc:.4f}")

    mean_auroc = sum(results.values()) / len(results)
    print(f"\n=========================================")
    print(f"Baseline 7 (DRAEM) Fast Mean AUROC: {mean_auroc:.4f}")
    print(f"=========================================")

Index missing! Run your dataset extraction cell first.


import os
import tarfile
import pandas as pd
from pathlib import Path

# Extract dataset from Drive
archive_path = "/content/drive/MyDrive/VisionBench_Datasets/mvtec_anomaly_detection.tar.xz"
extract_dir = "data/raw"

os.makedirs(extract_dir, exist_ok=True)
if os.path.exists(archive_path):
    print("Extracting dataset...")
    with tarfile.open(archive_path, "r:xz") as tar:
        tar.extractall(path=extract_dir)
    print("Extraction complete!")
else:
    print("Archive not found on Drive. Check Drive mounting.")

# Build Index CSV
records = []
data_path = Path("data/raw")
for train_dir in data_path.rglob("train"):
    cat_dir = train_dir.parent
    category = cat_dir.name
    good_train = train_dir / "good"
    if good_train.exists():
        for img_path in good_train.glob("*.png"):
            records.append({"category": category, "split": "train", "defect_type": "good", "image_path": str(img_path), "mask_path": "", "label": 0})
    test_dir = cat_dir / "test"
    gt_dir = cat_dir / "ground_truth"
    if test_dir.exists():
        for defect_dir in test_dir.iterdir():
            if defect_dir.is_dir():
                defect_type = defect_dir.name
                label = 0 if defect_type == "good" else 1
                for img_path in defect_dir.glob("*.png"):
                    mask_path_str = ""
                    if label == 1 and gt_dir.exists():
                        mask_file = gt_dir / defect_type / f"{img_path.stem}_mask.png"
                        if mask_file.exists():
                            mask_path_str = str(mask_file)
                    records.append({"category": category, "split": "test", "defect_type": defect_type, "image_path": str(img_path), "mask_path": mask_path_str, "label": label})

df = pd.DataFrame(records)
os.makedirs("data", exist_ok=True)
df.to_csv("data/mvtec_index.csv", index=False)
print(f"SUCCESS: Index created with {len(df)} images!")

import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import roc_auc_score

# 1. Model Definitions
class EncoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.conv1, self.bn1, self.relu, self.maxpool = resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4

    def forward(self, x):
        x0 = self.relu(self.bn1(self.conv1(x)))
        x1 = self.layer1(self.maxpool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return [x0, x1, x2, x3, x4]

class DecoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = nn.Sequential(nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU())
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU())
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec0 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())

        # Added an additional upsampling layer to reach 224x224
        self.up0 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2) # Upsamples 112x112 to 224x224
        self.final_conv = nn.Sequential(nn.Conv2d(32, 3, 3, padding=1), nn.Sigmoid()) # Final conv with corrected input channels

    def forward(self, feats):
        x0, x1, x2, x3, x4 = feats
        d4 = self.dec3(torch.cat([self.up4(x4), x3], dim=1))
        d3 = self.dec2(torch.cat([self.up3(d4), x2], dim=1))
        d2 = self.dec1(torch.cat([self.up2(d3), x1], dim=1))
        d1 = self.dec0(torch.cat([self.up1(d2), x0], dim=1))

        # Perform the final upsampling step
        d0 = self.up0(d1)
        return self.final_conv(d0)

class DiscriminatorSegment(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(6, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU()
        )
        self.head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, orig, recon):
        return self.head(self.stem(torch.cat([orig, recon], dim=1)))

class DRAEM(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = EncoderReconstruct()
        self.decoder = DecoderReconstruct()
        self.segmentor = DiscriminatorSegment()

    def forward(self, x):
        feats = self.encoder(x)
        recon = self.decoder(feats)
        return recon, self.segmentor(x, recon)

# 2. Noise & Dataset
def generate_perlin_noise_mask(height=224, width=224):
    scale = np.random.choice([16, 32, 64])
    h_small, w_small = max(1, height // scale), max(1, width // scale)
    noise = np.random.randn(h_small, w_small)
    noise = cv2.resize(noise, (width, height), interpolation=cv2.INTER_CUBIC)
    return torch.from_numpy((noise > 0.3).astype(np.float32)).unsqueeze(0)

class DRAEMTrainDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df[df['split'] == 'train'].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]['image_path']).convert("RGB")
        if self.transform: img = self.transform(img)
        mask = generate_perlin_noise_mask(img.shape[1], img.shape[2])
        aug = img * (1.0 - mask) + torch.rand_like(img) * mask
        return {"augmented_image": aug, "target_image": img, "anomaly_mask": mask}

# 3. Fast Runner
def train_and_eval_category(category_name, df, epochs=3, batch_size=8, lr=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cat_df = df[df['category'] == category_name]
    transform = T.Compose([T.Resize((224, 224)), T.ToTensor()])

    train_loader = DataLoader(DRAEMTrainDataset(cat_df, transform=transform), batch_size=batch_size, shuffle=True)
    model = DRAEM().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    l2_loss, bce_loss = nn.MSELoss(), nn.BCEWithLogitsLoss()

    model.train()
    for epoch in range(epochs):
        for batch in train_loader:
            aug, target, mask = batch['augmented_image'].to(device), batch['target_image'].to(device), batch['anomaly_mask'].to(device)
            optimizer.zero_grad()
            recon, seg_logits = model(aug)
            loss = l2_loss(recon, target) + 2.0 * bce_loss(seg_logits, mask)
            loss.backward()
            optimizer.step()

    model.eval()
    test_df = cat_df[cat_df['split'] == 'test']
    y_true, y_scores = [], []
    with torch.no_grad():
        for _, row in test_df.iterrows():
            img = Image.open(row['image_path']).convert("RGB")
            _, seg_logits = model(transform(img).unsqueeze(0).to(device))
            y_true.append(row['label'])
            y_scores.append(torch.max(torch.sigmoid(seg_logits)).item())

    return roc_auc_score(y_true, y_scores)

# Execution
df = pd.read_csv("data/mvtec_index.csv")
categories = df['category'].unique()
results = {}

print(f"Starting FAST 3-epoch DRAEM baseline run across {len(categories)} categories...\n")
for cat in categories:
    auroc = train_and_eval_category(cat, df, epochs=3)
    results[cat] = auroc
    print(f"Category: {cat:<12} | AUROC: {auroc:.4f}")

print(f"\n=========================================")
print(f"Baseline 7 (DRAEM) Fast Mean AUROC: {sum(results.values()) / len(results):.4f}")
print(f"=========================================")


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import roc_auc_score
from pathlib import Path

# Define PROJECT_ROOT explicitly for robust path handling
# Changed to point to the Google Drive location where data was extracted
PROJECT_ROOT = Path("/content/drive/MyDrive/VisionBench")

# 1. Model Definitions
class EncoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.conv1, self.bn1, self.relu, self.maxpool = resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4

    def forward(self, x):
        x0 = self.relu(self.bn1(self.conv1(x)))
        x1 = self.layer1(self.maxpool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return [x0, x1, x2, x3, x4]

class DecoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = nn.Sequential(nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU())
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU())
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec0 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.final = nn.Sequential(nn.Conv2d(64, 3, 3, padding=1), nn.Sigmoid())

    def forward(self, feats):
        x0, x1, x2, x3, x4 = feats
        d4 = self.dec3(torch.cat([self.up4(x4), x3], dim=1))
        d3 = self.dec2(torch.cat([self.up3(d4), x2], dim=1))
        d2 = self.dec1(torch.cat([self.up2(d3), x1], dim=1))
        d1 = self.dec0(torch.cat([self.up1(d2), x0], dim=1))
        return self.final(d1)

class DiscriminatorSegment(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(6, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU()
        )
        self.head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, orig, recon):
        return self.head(self.stem(torch.cat([orig, recon], dim=1)))

class DRAEM(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = EncoderReconstruct()
        self.decoder = DecoderReconstruct()
        self.segmentor = DiscriminatorSegment()

    def forward(self, x):
        feats = self.encoder(x)
        recon = self.decoder(feats)
        return recon, self.segmentor(x, recon)

# 2. Fast Noise & In-Memory Dataset (Caches images in RAM)
def generate_perlin_noise_mask(height=224, width=224):
    scale = np.random.choice([16, 32])
    h_small, w_small = max(1, height // scale), max(1, width // scale)
    noise = np.random.randn(h_small, w_small)
    noise = cv2.resize(noise, (width, height), interpolation=cv2.INTER_NEAREST)
    return torch.from_numpy((noise > 0.3).astype(np.float32)).unsqueeze(0)

class FastInMemoryDataset(Dataset):
    def __init__(self, df, transform=None):
        self.transform = transform
        train_df = df[df['split'] == 'train'].reset_index(drop=True)
        self.images = []
        for path in train_df['image_path']:
            # Use PROJECT_ROOT to ensure correct image path resolution
            img = Image.open(PROJECT_ROOT / path).convert("RGB")
            if transform: img = transform(img)
            self.images.append(img)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        mask = generate_perlin_noise_mask(img.shape[1], img.shape[2])
        aug = img * (1.0 - mask) + torch.rand_like(img) * mask
        return {"augmented_image": aug, "target_image": img, "anomaly_mask": mask}

# 3. Execution Function
def train_and_eval_fast(category_name, df, epochs=3, batch_size=16, lr=2e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cat_df = df[df['category'] == category_name]
    transform = T.Compose([T.Resize((224, 224)), T.ToTensor()])

    train_ds = FastInMemoryDataset(cat_df, transform=transform)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = DRAEM().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    l2_loss, bce_loss = nn.MSELoss(), nn.BCEWithLogitsLoss()

    model.train()
    for epoch in range(epochs):
        for batch in train_loader:
            aug, target, mask = batch['augmented_image'].to(device), batch['target_image'].to(device), batch['anomaly_mask'].to(device)
            optimizer.zero_grad()
            recon, seg_logits = model(aug)
            loss = l2_loss(recon, target) + 2.0 * bce_loss(seg_logits, mask)
            loss.backward()
            optimizer.step()

    model.eval()
    test_df = cat_df[cat_df['split'] == 'test']
    y_true, y_scores = [], []
    with torch.no_grad():
        for _, row in test_df.iterrows():
            img = Image.open(PROJECT_ROOT / row['image_path']).convert("RGB") # Use PROJECT_ROOT for test images too
            _, seg_logits = model(transform(img).unsqueeze(0).to(device))
            y_true.append(row['label'])
            y_scores.append(torch.max(torch.sigmoid(seg_logits)).item())

    return roc_auc_score(y_true, y_scores)
# Run Loop - using local path
index_path = "data/mvtec_index.csv"

if not os.path.exists(index_path):
    print(f"Error: {index_path} not found. Please run the extraction/indexing step first!")
else:
    df = pd.read_csv(index_path)
    categories = df['category'].unique()
    results = {}

    print(f"Starting FAST 3-epoch IN-MEMORY DRAEM run across {len(categories)} categories...\n")
    for cat in categories:
        auroc = train_and_eval_fast(cat, df, epochs=3, batch_size=16)
        results[cat] = auroc
        print(f"Category: {cat:<12} | AUROC: {auroc:.4f}")

    mean_auroc = sum(results.values()) / len(results)
    print(f"\n=========================================")
    print(f"Baseline 7 (DRAEM) Fast Mean AUROC: {mean_auroc:.4f}")
    print(f"=========================================")


Error: data/mvtec_index.csv not found. Please run the extraction/indexing step first!


In [ ]:
import os
import cv2
import tarfile
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import roc_auc_score

# ---------------------------------------------------------
# STEP 1: Re-extract Dataset & Build Local Index
# ---------------------------------------------------------
archive_path = "/content/drive/MyDrive/VisionBench_Datasets/mvtec_anomaly_detection.tar.xz"
extract_dir = "data/raw"

os.makedirs(extract_dir, exist_ok=True)
if os.path.exists(archive_path) and not os.path.exists("data/mvtec_index.csv"):
    print("Extracting dataset from Google Drive to local disk...")
    with tarfile.open(archive_path, "r:xz") as tar:
        tar.extractall(path=extract_dir)
    print("Extraction complete! Building index...")

    records = []
    data_path = Path("data/raw")
    for train_dir in data_path.rglob("train"):
        cat_dir = train_dir.parent
        category = cat_dir.name
        good_train = train_dir / "good"
        if good_train.exists():
            for img_path in good_train.glob("*.png"):
                records.append({"category": category, "split": "train", "defect_type": "good", "image_path": str(img_path), "mask_path": "", "label": 0})
        test_dir = cat_dir / "test"
        gt_dir = cat_dir / "ground_truth"
        if test_dir.exists():
            for defect_dir in test_dir.iterdir():
                if defect_dir.is_dir():
                    defect_type = defect_dir.name
                    label = 0 if defect_type == "good" else 1
                    for img_path in defect_dir.glob("*.png"):
                        mask_path_str = ""
                        if label == 1 and gt_dir.exists():
                            mask_file = gt_dir / defect_type / f"{img_path.stem}_mask.png"
                            if mask_file.exists():
                                mask_path_str = str(mask_file)
                        records.append({"category": category, "split": "test", "defect_type": defect_type, "image_path": str(img_path), "mask_path": mask_path_str, "label": label})

    df = pd.DataFrame(records)
    os.makedirs("data", exist_ok=True)
    df.to_csv("data/mvtec_index.csv", index=False)
    print(f"Index built successfully with {len(df)} images!\n")
else:
    print("Index already present. Skipping extraction.\n")

# ---------------------------------------------------------
# STEP 2: Fast In-Memory DRAEM Architecture & Training
# ---------------------------------------------------------
class EncoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.conv1, self.bn1, self.relu, self.maxpool = resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4

    def forward(self, x):
        x0 = self.relu(self.bn1(self.conv1(x)))
        x1 = self.layer1(self.maxpool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return [x0, x1, x2, x3, x4]

class DecoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = nn.Sequential(nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU())
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU())
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec0 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.final = nn.Sequential(nn.Conv2d(64, 3, 3, padding=1), nn.Sigmoid())

    def forward(self, feats):
        x0, x1, x2, x3, x4 = feats
        d4 = self.dec3(torch.cat([self.up4(x4), x3], dim=1))
        d3 = self.dec2(torch.cat([self.up3(d4), x2], dim=1))
        d2 = self.dec1(torch.cat([self.up2(d3), x1], dim=1))
        d1 = self.dec0(torch.cat([self.up1(d2), x0], dim=1))
        return self.final(d1)

class DiscriminatorSegment(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(6, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU()
        )
        self.head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, orig, recon):
        return self.head(self.stem(torch.cat([orig, recon], dim=1)))

class DRAEM(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = EncoderReconstruct()
        self.decoder = DecoderReconstruct()
        self.segmentor = DiscriminatorSegment()

    def forward(self, x):
        feats = self.encoder(x)
        recon = self.decoder(feats)
        return recon, self.segmentor(x, recon)

def generate_perlin_noise_mask(height=224, width=224):
    scale = np.random.choice([16, 32])
    h_small, w_small = max(1, height // scale), max(1, width // scale)
    noise = np.random.randn(h_small, w_small)
    noise = cv2.resize(noise, (width, height), interpolation=cv2.INTER_NEAREST)
    return torch.from_numpy((noise > 0.3).astype(np.float32)).unsqueeze(0)

class FastInMemoryDataset(Dataset):
    def __init__(self, df, transform=None):
        self.transform = transform
        train_df = df[df['split'] == 'train'].reset_index(drop=True)
        self.images = []
        for path in train_df['image_path']:
            img = Image.open(path).convert("RGB")
            if transform: img = transform(img)
            self.images.append(img)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        mask = generate_perlin_noise_mask(img.shape[1], img.shape[2])
        aug = img * (1.0 - mask) + torch.rand_like(img) * mask
        return {"augmented_image": aug, "target_image": img, "anomaly_mask": mask}

def train_and_eval_fast(category_name, df, epochs=3, batch_size=16, lr=2e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cat_df = df[df['category'] == category_name]
    transform = T.Compose([T.Resize((224, 224)), T.ToTensor()])

    train_ds = FastInMemoryDataset(cat_df, transform=transform)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = DRAEM().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    l2_loss, bce_loss = nn.MSELoss(), nn.BCEWithLogitsLoss()

    model.train()
    for epoch in range(epochs):
        for batch in train_loader:
            aug, target, mask = batch['augmented_image'].to(device), batch['target_image'].to(device), batch['anomaly_mask'].to(device)
            optimizer.zero_grad()
            recon, seg_logits = model(aug)
            loss = l2_loss(recon, target) + 2.0 * bce_loss(seg_logits, mask)
            loss.backward()
            optimizer.step()

    model.eval()
    test_df = cat_df[cat_df['split'] == 'test']
    y_true, y_scores = [], []
    with torch.no_grad():
        for _, row in test_df.iterrows():
            img = Image.open(row['image_path']).convert("RGB")
            _, seg_logits = model(transform(img).unsqueeze(0).to(device))
            y_true.append(row['label'])
            y_scores.append(torch.max(torch.sigmoid(seg_logits)).item())

    return roc_auc_score(y_true, y_scores)

# ---------------------------------------------------------
# STEP 3: Run Baseline Evaluation Loop
# ---------------------------------------------------------
df = pd.read_csv("data/mvtec_index.csv")
categories = df['category'].unique()
results = {}

print(f"Starting FAST 3-epoch IN-MEMORY DRAEM run across {len(categories)} categories...\n")
for cat in categories:
    auroc = train_and_eval_fast(cat, df, epochs=3, batch_size=16)
    results[cat] = auroc
    print(f"Category: {cat:<12} | AUROC: {auroc:.4f}")

mean_auroc = sum(results.values()) / len(results)
print(f"\n=========================================")
print(f"Baseline 7 (DRAEM) Fast Mean AUROC: {mean_auroc:.4f}")
print(f"=========================================")

Index already present. Skipping extraction.



FileNotFoundError: [Errno 2] No such file or directory: 'data/mvtec_index.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import tarfile
import pandas as pd
from pathlib import Path

archive_path = "/content/drive/MyDrive/VisionBench_Datasets/mvtec_anomaly_detection.tar.xz"
extract_dir = "data/raw"

os.makedirs(extract_dir, exist_ok=True)
if os.path.exists(archive_path):
    print("Extracting dataset from Google Drive...")
    with tarfile.open(archive_path, "r:xz") as tar:
        tar.extractall(path=extract_dir)
    print("Extraction complete! Building index...")

    records = []
    data_path = Path("data/raw")
    for train_dir in data_path.rglob("train"):
        cat_dir = train_dir.parent
        category = cat_dir.name
        good_train = train_dir / "good"
        if good_train.exists():
            for img_path in good_train.glob("*.png"):
                records.append({"category": category, "split": "train", "defect_type": "good", "image_path": str(img_path), "mask_path": "", "label": 0})
        test_dir = cat_dir / "test"
        gt_dir = cat_dir / "ground_truth"
        if test_dir.exists():
            for defect_dir in test_dir.iterdir():
                if defect_dir.is_dir():
                    defect_type = defect_dir.name
                    label = 0 if defect_type == "good" else 1
                    for img_path in defect_dir.glob("*.png"):
                        mask_path_str = ""
                        if label == 1 and gt_dir.exists():
                            mask_file = gt_dir / defect_type / f"{img_path.stem}_mask.png"
                            if mask_file.exists():
                                mask_path_str = str(mask_file)
                        records.append({"category": category, "split": "test", "defect_type": defect_type, "image_path": str(img_path), "mask_path": mask_path_str, "label": label})

    df = pd.DataFrame(records)
    os.makedirs("data", exist_ok=True)
    df.to_csv("data/mvtec_index.csv", index=False)
    print(f"SUCCESS: Index created with {len(df)} images!")
else:
    print(f"ERROR: Archive not found at {archive_path}. Check your Drive path.")

Extracting dataset from Google Drive...


/tmp/ipykernel_2313/1471985578.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


Extraction complete! Building index...
SUCCESS: Index created with 5354 images!


In [ ]:
import os
import cv2
import tarfile
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import roc_auc_score
from google.colab import drive

# 1. Mount Drive & Ensure Index Exists
drive.mount('/content/drive', force_remount=False)

index_path = "data/mvtec_index.csv"
archive_path = "/content/drive/MyDrive/VisionBench_Datasets/mvtec_anomaly_detection.tar.xz"

if not os.path.exists(index_path):
    print("Extracting dataset and creating local index...")
    extract_dir = "data/raw"
    os.makedirs(extract_dir, exist_ok=True)
    with tarfile.open(archive_path, "r:xz") as tar:
        tar.extractall(path=extract_dir)

    records = []
    data_path = Path("data/raw")
    for train_dir in data_path.rglob("train"):
        cat_dir = train_dir.parent
        category = cat_dir.name
        good_train = train_dir / "good"
        if good_train.exists():
            for img_path in good_train.glob("*.png"):
                records.append({"category": category, "split": "train", "defect_type": "good", "image_path": str(img_path), "mask_path": "", "label": 0})
        test_dir = cat_dir / "test"
        gt_dir = cat_dir / "ground_truth"
        if test_dir.exists():
            for defect_dir in test_dir.iterdir():
                if defect_dir.is_dir():
                    defect_type = defect_dir.name
                    label = 0 if defect_type == "good" else 1
                    for img_path in defect_dir.glob("*.png"):
                        mask_path_str = ""
                        if label == 1 and gt_dir.exists():
                            mask_file = gt_dir / defect_type / f"{img_path.stem}_mask.png"
                            if mask_file.exists():
                                mask_path_str = str(mask_file)
                        records.append({"category": category, "split": "test", "defect_type": defect_type, "image_path": str(img_path), "mask_path": mask_path_str, "label": label})

    df = pd.DataFrame(records)
    os.makedirs("data", exist_ok=True)
    df.to_csv(index_path, index=False)
    print(f"Index created with {len(df)} images.")
else:
    df = pd.read_csv(index_path)
    print(f"Index loaded with {len(df)} images.")

# 2. Architecture Definitions
class EncoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.conv1, self.bn1, self.relu, self.maxpool = resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4

    def forward(self, x):
        x0 = self.relu(self.bn1(self.conv1(x)))
        x1 = self.layer1(self.maxpool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return [x0, x1, x2, x3, x4]

class DecoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = nn.Sequential(nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU()) # Output 14x14
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU()) # Output 28x28
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU()) # Output 56x56
        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2) # Output 112x112
        self.dec0 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU()) # Output 112x112

        # Added an additional upsampling layer to reach 224x224
        self.up0 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2) # Upsamples 112x112 to 224x224
        self.final_conv = nn.Sequential(nn.Conv2d(32, 3, 3, padding=1), nn.Sigmoid()) # Final conv with corrected input channels

    def forward(self, feats):
        x0, x1, x2, x3, x4 = feats
        d4 = self.dec3(torch.cat([self.up4(x4), x3], dim=1))
        d3 = self.dec2(torch.cat([self.up3(d4), x2], dim=1))
        d2 = self.dec1(torch.cat([self.up2(d3), x1], dim=1))
        d1 = self.dec0(torch.cat([self.up1(d2), x0], dim=1))

        # Perform the final upsampling step
        d0 = self.up0(d1)
        return self.final_conv(d0)

class DiscriminatorSegment(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(6, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU()
        )
        self.head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, orig, recon):
        return self.head(self.stem(torch.cat([orig, recon], dim=1)))

class DRAEM(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = EncoderReconstruct()
        self.decoder = DecoderReconstruct()
        self.segmentor = DiscriminatorSegment()

    def forward(self, x):
        feats = self.encoder(x)
        recon = self.decoder(feats)
        return recon, self.segmentor(x, recon)

# 3. Noise Generation & Fast RAM Dataset
def generate_perlin_noise_mask(height=224, width=224):
    scale = np.random.choice([16, 32])
    h_small, w_small = max(1, height // scale), max(1, width // scale)
    noise = np.random.randn(h_small, w_small)
    noise = cv2.resize(noise, (width, height), interpolation=cv2.INTER_NEAREST)
    return torch.from_numpy((noise > 0.3).astype(np.float32)).unsqueeze(0)

class FastInMemoryDataset(Dataset):
    def __init__(self, df, transform=None):
        self.transform = transform
        train_df = df[df['split'] == 'train'].reset_index(drop=True)
        self.images = []
        # Pre-loads into system RAM to bypass disk read bottlenecks
        for path in train_df['image_path']:
            img = Image.open(path).convert("RGB")
            if transform: img = transform(img)
            self.images.append(img)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        mask = generate_perlin_noise_mask(img.shape[1], img.shape[2])
        aug = img * (1.0 - mask) + torch.rand_like(img) * mask
        return {"augmented_image": aug, "target_image": img, "anomaly_mask": mask}

# 4. Fast Trainer Function
def train_and_eval_fast(category_name, df, epochs=3, batch_size=16, lr=2e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cat_df = df[df['category'] == category_name]
    transform = T.Compose([T.Resize((224, 224)), T.ToTensor()])

    train_ds = FastInMemoryDataset(cat_df, transform=transform)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = DRAEM().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    l2_loss, bce_loss = nn.MSELoss(), nn.BCEWithLogitsLoss()

    model.train()
    for epoch in range(epochs):
        for batch in train_loader:
            aug, target, mask = batch['augmented_image'].to(device), batch['target_image'].to(device), batch['anomaly_mask'].to(device)
            optimizer.zero_grad()
            recon, seg_logits = model(aug)
            loss = l2_loss(recon, target) + 2.0 * bce_loss(seg_logits, mask)
            loss.backward()
            optimizer.step()

    model.eval()
    test_df = cat_df[cat_df['split'] == 'test']
    y_true, y_scores = [], []
    with torch.no_grad():
        for _, row in test_df.iterrows():
            img = Image.open(row['image_path']).convert("RGB")
            _, seg_logits = model(transform(img).unsqueeze(0).to(device))
            y_true.append(row['label'])
            y_scores.append(torch.max(torch.sigmoid(seg_logits)).item())

    return roc_auc_score(y_true, y_scores)

# 5. Execution Loop across 15 categories
categories = df['category'].unique()
results = {}

print(f"Starting FAST IN-MEMORY DRAEM run across {len(categories)} categories...\n")
for cat in categories:
    auroc = train_and_eval_fast(cat, df, epochs=3, batch_size=16)
    results[cat] = auroc
    print(f"Category: {cat:<12} | AUROC: {auroc:.4f}")

mean_auroc = sum(results.values()) / len(results)
print(f"\n=========================================")
print(f"Baseline 7 (DRAEM) Fast Mean AUROC: {mean_auroc:.4f}")
print(f"=========================================")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Index loaded with 5354 images.
Starting FAST IN-MEMORY DRAEM run across 15 categories...

Category: zipper       | AUROC: 0.5499
Category: grid         | AUROC: 0.8429
Category: tile         | AUROC: 0.5693
Category: capsule      | AUROC: 0.5417
Category: screw        | AUROC: 0.3775
Category: leather      | AUROC: 0.6464
Category: carpet       | AUROC: 0.6096
Category: metal_nut    | AUROC: 0.3915
Category: toothbrush   | AUROC: 0.2778
Category: bottle       | AUROC: 0.6071
Category: transistor   | AUROC: 0.4658
Category: wood         | AUROC: 0.7018
Category: hazelnut     | AUROC: 0.8871
Category: pill         | AUROC: 0.4433
Category: cable        | AUROC: 0.5238

Baseline 7 (DRAEM) Fast Mean AUROC: 0.5624


In [ ]:
import pandas as pd
import torch
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import os
import numpy as np
import cv2
from PIL import Image
import torchvision.models as models
from pathlib import Path

# Quick 1-Category Loss Check (Takes ~10 seconds)
cat_name = "hazelnut"

# Explicitly define PROJECT_ROOT to ensure correct path resolution
PROJECT_ROOT = Path('/content/drive/MyDrive/VisionBench')

# Ensure df is loaded from the index file using the absolute path
df = pd.read_csv(PROJECT_ROOT / "data" / "mvtec_index.csv")

# Ensure device is defined
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Model Definitions (Copied for self-containment) ---
class EncoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.conv1, self.bn1, self.relu, self.maxpool = resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4

    def forward(self, x):
        x0 = self.relu(self.bn1(self.conv1(x)))
        x1 = self.layer1(self.maxpool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return [x0, x1, x2, x3, x4]

class DecoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = nn.Sequential(nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU()) # Output 14x14
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU()) # Output 28x28
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU()) # Output 56x56
        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2) # Output 112x112
        self.dec0 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU()) # Output 112x112

        # Added an additional upsampling layer to reach 224x224
        self.up0 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2) # Upsamples 112x112 to 224x224
        self.final_conv = nn.Sequential(nn.Conv2d(32, 3, 3, padding=1), nn.Sigmoid()) # Final conv with corrected input channels

    def forward(self, feats):
        x0, x1, x2, x3, x4 = feats
        d4 = self.dec3(torch.cat([self.up4(x4), x3], dim=1))
        d3 = self.dec2(torch.cat([self.up3(d4), x2], dim=1))
        d2 = self.dec1(torch.cat([self.up2(d3), x1], dim=1))
        d1 = self.dec0(torch.cat([self.up1(d2), x0], dim=1))

        # Perform the final upsampling step
        d0 = self.up0(d1)
        return self.final_conv(d0)

class DiscriminatorSegment(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(6, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU()
        )
        self.head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, orig, recon):
        return self.head(self.stem(torch.cat([orig, recon], dim=1)))

class DRAEM(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = EncoderReconstruct()
        self.decoder = DecoderReconstruct()
        self.segmentor = DiscriminatorSegment()

    def forward(self, x):
        feats = self.encoder(x)
        recon = self.decoder(feats)
        return recon, self.segmentor(x, recon)

# --- Noise Generation & Dataset (Copied for self-containment) ---
def generate_perlin_noise_mask(height=224, width=224):
    scale = np.random.choice([16, 32])
    h_small, w_small = max(1, height // scale), max(1, width // scale)
    noise = np.random.randn(h_small, w_small)
    noise = cv2.resize(noise, (width, height), interpolation=cv2.INTER_NEAREST)
    return torch.from_numpy((noise > 0.3).astype(np.float32)).unsqueeze(0)

class FastInMemoryDataset(Dataset):
    def __init__(self, df, transform=None):
        self.transform = transform
        train_df = df[df['split'] == 'train'].reset_index(drop=True)
        self.images = []
        # Pre-loads into system RAM to bypass disk read bottlenecks
        for path in train_df['image_path']:
            # Use PROJECT_ROOT to ensure correct image path resolution
            img = Image.open(PROJECT_ROOT / path).convert("RGB")
            if transform: img = transform(img)
            self.images.append(img)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        mask = generate_perlin_noise_mask(img.shape[1], img.shape[2])
        aug = img * (1.0 - mask) + torch.rand_like(img) * mask
        return {"augmented_image": aug, "target_image": img, "anomaly_mask": mask}

# --- End of Copied Definitions ---

cat_df = df[df['category'] == cat_name]
transform = T.Compose([T.Resize((224, 224)), T.ToTensor()])

train_ds = FastInMemoryDataset(cat_df, transform=transform)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

model = DRAEM().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
l2_loss, bce_loss = nn.MSELoss(), nn.BCEWithLogitsLoss()

print(f"--- Training Loss Check for [{cat_name}] ---")
model.train()
for epoch in range(3):
    total_loss, rec_loss, seg_loss = 0.0, 0.0, 0.0
    for batch in train_loader:
        aug, target, mask = batch['augmented_image'].to(device), batch['target_image'].to(device), batch['anomaly_mask'].to(device)
        optimizer.zero_grad()
        recon, seg_logits = model(aug)

        l_rec = l2_loss(recon, target)
        l_seg = bce_loss(seg_logits, mask)
        loss = l_rec + 2.0 * l_seg

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        rec_loss += l_rec.item()
        seg_loss += l_seg.item()

    num_batches = len(train_loader)
    print(f"Epoch {epoch+1}/3 | Total Loss: {total_loss/num_batches:.4f} | L_rec: {rec_loss/num_batches:.4f} | L_seg: {seg_loss/num_batches:.4f}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 125MB/s]


--- Training Loss Check for [hazelnut] ---
Epoch 1/3 | Total Loss: 0.8043 | L_rec: 0.0824 | L_seg: 0.3609
Epoch 2/3 | Total Loss: 0.4902 | L_rec: 0.0351 | L_seg: 0.2276
Epoch 3/3 | Total Loss: 0.4148 | L_rec: 0.0093 | L_seg: 0.2028


In [ ]:
import os
import tarfile
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset # Import Dataset
import torchvision.transforms as T
import torchvision.models as models
from google.colab import drive
import numpy as np # Import numpy for perlin noise
import cv2 # Import cv2 for perlin noise
from sklearn.metrics import roc_auc_score # Needed for evaluation, though not directly in this quick check, it's good practice to have it with DRAEM

# 1. Ensure Drive & Data Index are Present
drive.mount('/content/drive', force_remount=False)

index_path = "data/mvtec_index.csv"
archive_path = "/content/drive/MyDrive/VisionBench_Datasets/mvtec_anomaly_detection.tar.xz"

# Dynamic PROJECT_ROOT for consistent path handling
PROJECT_ROOT = Path('/content/drive/MyDrive/VisionBench')

if not os.path.exists(index_path) or os.path.getsize(index_path) == 0:
    print("Re-extracting dataset and creating local index...")
    extract_dir = "data/raw"
    os.makedirs(extract_dir, exist_ok=True)

    # Ensure archive exists before attempting extraction
    if not os.path.exists(archive_path):
        print(f"Error: Archive not found at {archive_path}. Please upload the dataset archive.")
    else:
        with tarfile.open(archive_path, "r:xz") as tar:
            tar.extractall(path=extract_dir)

        records = []
        data_path = Path("data/raw")
        for train_dir in data_path.rglob("train"):
            cat_dir = train_dir.parent
            category = cat_dir.name
            good_train = train_dir / "good"
            if good_train.exists():
                for img_path in good_train.glob("*.png"):
                    # Using relative path to PROJECT_ROOT for image_path
                    records.append({"category": category, "split": "train", "defect_type": "good", "image_path": str(Path(img_path).relative_to(PROJECT_ROOT)), "mask_path": "", "label": 0})

        # Add test data processing to the index creation for completeness
        for cat_dir in data_path.iterdir():
            if cat_dir.is_dir():
                category = cat_dir.name
                test_dir = cat_dir / "test"
                gt_dir = cat_dir / "ground_truth"

                if test_dir.exists():
                    for defect_dir in test_dir.iterdir():
                        if defect_dir.is_dir():
                            defect_type = defect_dir.name
                            label = 0 if defect_type == "good" else 1
                            for img_path in defect_dir.glob("*.png"):
                                mask_path_str = ""
                                if label == 1 and gt_dir.exists():
                                    mask_file = gt_dir / defect_type / f"{img_path.stem}_mask.png"
                                    if mask_file.exists():
                                        mask_path_str = str(Path(mask_file).relative_to(PROJECT_ROOT))
                                records.append({"category": category, "split": "test", "defect_type": defect_type, "image_path": str(Path(img_path).relative_to(PROJECT_ROOT)), "mask_path": mask_path_str, "label": label})

        df = pd.DataFrame(records)
        os.makedirs("data", exist_ok=True)
        df.to_csv(index_path, index=False)
        print(f"Index created with {len(df)} images.")
else:
    df = pd.read_csv(index_path)
    print(f"Index loaded with {len(df)} images.")

# --- Model Definitions (Copied for self-containment) ---
class EncoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.conv1, self.bn1, self.relu, self.maxpool = resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4

    def forward(self, x):
        x0 = self.relu(self.bn1(self.conv1(x)))
        x1 = self.layer1(self.maxpool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return [x0, x1, x2, x3, x4]

class DecoderReconstruct(nn.Module):
    def __init__(self):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = nn.Sequential(nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU()) # Output 14x14
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU()) # Output 28x28
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU()) # Output 56x56
        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2) # Output 112x112
        self.dec0 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU()) # Output 112x112

        # Added an additional upsampling layer to reach 224x224
        self.up0 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2) # Upsamples 112x112 to 224x224
        self.final_conv = nn.Sequential(nn.Conv2d(32, 3, 3, padding=1), nn.Sigmoid()) # Final conv with corrected input channels

    def forward(self, feats):
        x0, x1, x2, x3, x4 = feats
        d4 = self.dec3(torch.cat([self.up4(x4), x3], dim=1))
        d3 = self.dec2(torch.cat([self.up3(d4), x2], dim=1))
        d2 = self.dec1(torch.cat([self.up2(d3), x1], dim=1))
        d1 = self.dec0(torch.cat([self.up1(d2), x0], dim=1))

        # Perform the final upsampling step
        d0 = self.up0(d1)
        return self.final_conv(d0)

class DiscriminatorSegment(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(6, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU()
        )
        self.head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, orig, recon):
        return self.head(self.stem(torch.cat([orig, recon], dim=1)))

class DRAEM(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = EncoderReconstruct()
        self.decoder = DecoderReconstruct()
        self.segmentor = DiscriminatorSegment()

    def forward(self, x):
        feats = self.encoder(x)
        recon = self.decoder(feats)
        return recon, self.segmentor(x, recon)

# --- Noise Generation & Dataset (Copied for self-containment) ---
def generate_perlin_noise_mask(height=224, width=224):
    scale = np.random.choice([16, 32])
    h_small, w_small = max(1, height // scale), max(1, width // scale)
    noise = np.random.randn(h_small, w_small)
    noise = cv2.resize(noise, (width, height), interpolation=cv2.INTER_NEAREST)
    return torch.from_numpy((noise > 0.3).astype(np.float32)).unsqueeze(0)

class FastInMemoryDataset(Dataset):
    def __init__(self, df, transform=None):
        self.transform = transform
        train_df = df[df['split'] == 'train'].reset_index(drop=True)
        self.images = []
        # Pre-loads into system RAM to bypass disk read bottlenecks
        for path in train_df['image_path']:
            # Use PROJECT_ROOT to ensure correct image path resolution
            img = Image.open(PROJECT_ROOT / path).convert("RGB")
            if transform: img = transform(img)
            self.images.append(img)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        mask = generate_perlin_noise_mask(img.shape[1], img.shape[2])
        aug = img * (1.0 - mask) + torch.rand_like(img) * mask
        return {"augmented_image": aug, "target_image": img, "anomaly_mask": mask}

# --- End of Copied Definitions ---

# 2. Run Single Category Loss Execution
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cat_name = "hazelnut"
cat_df = df[df['category'] == cat_name]
transform = T.Compose([T.Resize((224, 224)), T.ToTensor()])

train_ds = FastInMemoryDataset(cat_df, transform=transform)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

model = DRAEM().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
l2_loss, bce_loss = nn.MSELoss(), nn.BCEWithLogitsLoss()

print(f"--- Training Loss Metrics for [{cat_name}] ---", flush=True)
model.train()
for epoch in range(3):
    total_loss, rec_loss, seg_loss = 0.0, 0.0, 0.0
    for batch in train_loader:
        aug, target, mask = batch['augmented_image'].to(device), batch['target_image'].to(device), batch['anomaly_mask'].to(device)
        optimizer.zero_grad()
        recon, seg_logits = model(aug)

        l_rec = l2_loss(recon, target)
        l_seg = bce_loss(seg_logits, mask)
        loss = l_rec + 2.0 * l_seg

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        rec_loss += l_rec.item()
        seg_loss += l_seg.item()

    num_batches = len(train_loader)
    print(f"Epoch {epoch+1}/3 | Total Loss: {total_loss/num_batches:.4f} | L_rec: {rec_loss/num_batches:.4f} | L_seg: {seg_loss/num_batches:.4f}", flush=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Index loaded with 3629 images.
--- Training Loss Metrics for [hazelnut] ---
Epoch 1/3 | Total Loss: 0.6568 | L_rec: 0.0786 | L_seg: 0.2891
Epoch 2/3 | Total Loss: 0.3920 | L_rec: 0.0314 | L_seg: 0.1803
Epoch 3/3 | Total Loss: 0.3301 | L_rec: 0.0135 | L_seg: 0.1583
